In [ ]:
'''
Filter and parse PubMed search results
'''

from collections import Counter
file = 'data/pubmed-ai_leukemia.txt'  # results file in PubMed format   

In [ ]:
'''
Create a dictionary to look up articles by PMID
'''
from collections import defaultdict
pmids = {}
in_abstract = False
in_title = False
with open(f) as infile:
    for line in infile:
        orig_line = line
        line = line.strip()
        if line.startswith('PMID-'):
            pmid = line.split()[1]
            pmids[pmid] = defaultdict(list)
        elif line.startswith('TI  -'):
            title = line.split('TI  - ')[1]
            pmids[pmid]['title'] = title
            in_title = True                
        elif line.startswith('MH  -'):
            title = line.split('MH  - ')[1]
            pmids[pmid]['MH'] += [title]
        elif line.startswith('LA  -'):
            language = line.split('LA  - ')[1]
            pmids[pmid]['LA'] += [language]    
        elif line.startswith('AB  -'):            
            abstract = line.split('AB  - ')[1] + ' '
            in_abstract = True
        elif line.startswith('PT  -'):
            pub_type = line.split('PT  - ')[1]
            pmids[pmid]['PT'] += [pub_type]
        elif line.startswith('JT  -'):
            jt = line.split('JT  - ')[1]
            pmids[pmid]['JT'] += [jt]
        elif orig_line.startswith(' '):             
            if in_abstract :
                abstract += ' ' + line
            elif in_title :
                title += ' ' + line
        elif not orig_line.startswith(' '):
            if in_abstract :
                pmids[pmid]['AB'] = abstract + ' '
                in_abstract = False
            elif in_title :
                pmids[pmid]['title'] = title
                in_title = False
        

In [ ]:
def countPTs(pmids, type = 'PT'):
    '''Count number of occurences of each type'''
    print("Number of PMIDs: ", len(pmids))
    pub_types = [pt for key in pmids for pt in pmids[key][type] ]
    return Counter(pub_types)

countPTs(pmids)

In [ ]:
def filter_pmids(pmids, skip, type = 'PT'):
    ''' filter pmid dictionary by removing articles with any skip value in type'''
    pmids2 = {key: val for key, val in pmids.items() if not any([skip in pt.lower() for pt in pmids[key][type]])}
    return pmids2

In [ ]:
# filter to remove article types
pmids = filter_pmids(pmids, 'review')
pmids = filter_pmids(pmids, 'letter')
pmids = filter_pmids(pmids, 'comment')
pmids = filter_pmids(pmids, 'editorial')
pmids = filter_pmids(pmids, 'retracted')
pmids = filter_pmids(pmids, 'biography')
countPTs(pmids, 'PT')



In [ ]:
# filter to remove non-English articles
pmids = filter_pmids(pmids, type = 'LA', skip = 'jpn')
pmids = filter_pmids(pmids, type = 'LA', skip = 'ger')
countPTs(pmids, 'LA')

In [ ]:
# Copy results to tsv file
with open('articles.tsv', 'w') as outfile:
    outfile.write('PMID\tTitle\tAbstract\tJournal\n')
    for pmid in pmids:
        article = pmids[pmid]
        outfile.write(f"{pmid}\t{article['title']}\t{article['AB']}\t{article['JT']}\n")